<a href="https://colab.research.google.com/github/Kryptera-K/ORCL-Adaptive-EMA-QStick-Hybrid-Strategy/blob/main/ORCL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install vectorbt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.7/527.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import vectorbt as vbt

# -------------------------
# Download Data
# -------------------------

symbol = "ORCL"
start_date = "2000-01-01"
end_date = "2026-01-01"
interval = "1d"

df = yf.download(symbol, start=start_date, end=end_date, interval=interval, multi_level_index=False)
df.to_csv("ORCL_clean.csv", index=False)
df

/tmp/ipython-input-1575473172.py:15: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed


,Close,High,Low,Open,Volume
Date,,,,,
2000-01-03,23.673817,25.089235,22.371130,24.976503,98114800
2000-01-04,21.581999,23.774018,21.043388,23.147727,116824800
2000-01-05,20.442156,21.318964,19.239676,20.367001,166054000
2000-01-06,19.239679,21.043399,18.976637,20.072647,109880000
2000-01-07,20.717710,20.742762,18.751156,19.039250,91755600
...,...,...,...,...,...
2025-10-17,291.309998,304.279999,287.500000,303.750000,37653000
2025-10-20,277.179993,289.239990,275.309998,288.950012,32810700
2025-10-21,275.149994,280.299988,272.260010,278.109985,18370800


In [ ]:
# -------------------------
# Necessary Parameters
# -------------------------

BULL_EMA_PERIOD = 13
BULL_LEVEL = 0
EMA_PERIOD = 20
EMA_SHIFT = 5
QSTICK_LEVEL = 0
QSTICK_PERIOD = 10
STOCHASTIC_D_PERIOD = 3
STOCHASTIC_K_PERIOD = 14
STOCHASTIC_OVERBOUGHT = 80
STOCHASTIC_OVERSOLD = 20

# -------------------------
# Indicator Functions
# -------------------------

def open_below_ema(df, period=EMA_PERIOD):
    df = calculate_ema(df, period)
    return df['Open'] < df['EMA']


def calculate_ema(df, period=EMA_PERIOD):
    """
    Calculate Exponential Moving Average (EMA) of the Close price.
    """
    df = df.copy()
    df['EMA'] = df['Close'].ewm(span=period, adjust=False).mean()
    return df


def qstick_cross_below_level(df, level=QSTICK_LEVEL):
    df = qstick(df)
    qs = df['QStick']
    return (qs < level) & (qs.shift(1) >= level)


def qstick(df, period=QSTICK_PERIOD):
    df = df.copy()
    df['QStick'] = (df['Close'] - df['Open']).rolling(period).mean()
    return df


def bull_power_higher_than(df, EMA_PERIOD=BULL_EMA_PERIOD, level=BULL_LEVEL):
    df = calculate_bull_power(df, EMA_PERIOD)
    return df['Bull_Power'] > level


def calculate_bull_power(df, EMA_PERIOD=BULL_EMA_PERIOD):
    """
    Calculate Bull Power indicator.
    Bull Power = High - EMA(Close)
    """
    df = df.copy()
    df['EMA'] = df['Close'].ewm(span=EMA_PERIOD, adjust=False).mean()
    df['Bull_Power'] = df['High'] - df['EMA']
    return df


def fast_k_cross_below_oversold(df, level=STOCHASTIC_OVERSOLD):
    df = calculate_stochastic(df)
    return (df['Fast_%K'].shift(1) > level) & (df['Fast_%K'] < level)


def calculate_stochastic(df, k_period=STOCHASTIC_K_PERIOD, d_period=STOCHASTIC_D_PERIOD):
    """Calculate Fast %K and Slow %D"""
    low_min = df['Low'].rolling(window=k_period).min()
    high_max = df['High'].rolling(window=k_period).max()

    df['Fast_%K'] = 100 * (df['Close'] - low_min) / (high_max - low_min)
    df['Slow_%D'] = df['Fast_%K'].rolling(window=d_period).mean()
    return df



# -------------------------
# Entry conditions
# -------------------------

df["EMA_Open_Below"] = open_below_ema(df)
df["QStick_Cross_Below_Zero"] = qstick_cross_below_level(df)

# -------------------------
# Exit conditions
# -------------------------

df["BullP_Higher_0"] = bull_power_higher_than(df)
df["Stochastic_Fast_%K_Cross_Below_Oversold"] = fast_k_cross_below_oversold(df)

# -------------------------
# Signals
# -------------------------

entry_conditions = [
    'EMA_Open_Below',
    'QStick_Cross_Below_Zero',
]
exit_conditions = [
    'BullP_Higher_0',
    'Stochastic_Fast_%K_Cross_Below_Oversold',
]

df['entry_signal'] = df[entry_conditions].all(axis=1)
df['exit_signal']  = df[exit_conditions].all(axis=1)

# -------------------------
# Backtest
# -------------------------


shift_entries = df['entry_signal'].shift(1).astype(bool).fillna(False).to_numpy()
shift_exits = df['exit_signal'].shift(1).astype(bool).fillna(False).to_numpy()

pf = vbt.Portfolio.from_signals(
    close=df['Open'],
    entries=shift_entries,
    exits=shift_exits,
    init_cash=100_000,
    fees=0.001,
    slippage=0.002,
    freq='1d'
)


# -------------------------
# Portfolio Stats / Plot
# -------------------------

print(pf.stats())
pf.plot().show()

Start                                2000-01-03 00:00:00
End                                  2025-10-23 00:00:00
Period                                6492 days 00:00:00
Start Value                                     100000.0
End Value                                 2392922.208555
Total Return [%]                             2292.922209
Benchmark Return [%]                          992.867399
Max Gross Exposure [%]                             100.0
Total Fees Paid                             49274.319966
Max Drawdown [%]                               59.518637
Max Drawdown Duration                 2092 days 00:00:00
Total Trades                                          67
Total Closed Trades                                   67
Total Open Trades                                      0
Open Trade PnL                                       0.0
Win Rate [%]                                   58.208955
Best Trade [%]                                 67.965435
Worst Trade [%]                

In [ ]:
# Buy and Hold Performance Metrics
df_holding = df['Open']
pf_holding = vbt.Portfolio.from_holding(df_holding, init_cash=100_000 , freq='D')
print(pf_holding.stats())

Start                         2000-01-03 00:00:00
End                           2025-10-23 00:00:00
Period                         6492 days 00:00:00
Start Value                              100000.0
End Value                           1092867.39875
Total Return [%]                       992.867399
Benchmark Return [%]                   992.867399
Max Gross Exposure [%]                      100.0
Total Fees Paid                               0.0
Max Drawdown [%]                        84.000004
Max Drawdown Duration          3593 days 00:00:00
Total Trades                                    1
Total Closed Trades                             0
Total Open Trades                               1
Open Trade PnL                       992867.39875
Win Rate [%]                                  NaN
Best Trade [%]                                NaN
Worst Trade [%]                               NaN
Avg Winning Trade [%]                         NaN
Avg Losing Trade [%]                          NaN


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -------------------------
# Download Data
# -------------------------

symbol = "ORCL"
start_date = "2020-01-01"
end_date = "2026-01-01"
interval = "1d"

df = yf.download(symbol, start=start_date, end=end_date, interval=interval, multi_level_index=False)

# -------------------------
# Parameters
# -------------------------

BULL_EMA_PERIOD = 13
BULL_LEVEL = 0
EMA_PERIOD = 20
QSTICK_PERIOD = 10
QSTICK_LEVEL = 0
STOCHASTIC_D_PERIOD = 3
STOCHASTIC_K_PERIOD = 14
STOCHASTIC_OVERBOUGHT = 80
STOCHASTIC_OVERSOLD = 20

# -------------------------
# Indicator Functions
# -------------------------

def calculate_ema(df, period=EMA_PERIOD):
    df = df.copy()
    df['EMA'] = df['Close'].ewm(span=period, adjust=False).mean()
    return df

def qstick(df, period=QSTICK_PERIOD):
    df = df.copy()
    df['QStick'] = (df['Close'] - df['Open']).rolling(period).mean()
    return df

def calculate_bull_power(df, EMA_PERIOD=BULL_EMA_PERIOD):
    df = df.copy()
    df['EMA_Bull'] = df['Close'].ewm(span=EMA_PERIOD, adjust=False).mean()
    df['Bull_Power'] = df['High'] - df['EMA_Bull']
    return df

def calculate_stochastic(df, k_period=STOCHASTIC_K_PERIOD, d_period=STOCHASTIC_D_PERIOD):
    df = df.copy()
    low_min = df['Low'].rolling(window=k_period).min()
    high_max = df['High'].rolling(window=k_period).max()
    df['Fast_%K'] = 100 * (df['Close'] - low_min) / (high_max - low_min)
    df['Slow_%D'] = df['Fast_%K'].rolling(window=d_period).mean()
    return df

# -------------------------
# Apply Indicators
# -------------------------

df = calculate_ema(df)
df = qstick(df)
df = calculate_bull_power(df)
df = calculate_stochastic(df)

# -------------------------
# Create Subplots
# -------------------------

fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.5, 0.15, 0.15, 0.2],
    subplot_titles=("Price with EMA", "QStick", "Bull Power", "Stochastic Oscillator")
)

# --- Price + EMA ---
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'],
    name='Candlestick'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['EMA'], mode='lines', name=f'EMA({EMA_PERIOD})',
    line=dict(width=1.5)
), row=1, col=1)

# --- QStick ---
fig.add_trace(go.Scatter(
    x=df.index, y=df['QStick'], mode='lines', name=f'QStick({QSTICK_PERIOD})',
    line=dict(width=1.5, color='orange')
), row=2, col=1)

fig.add_hline(y=QSTICK_LEVEL, line_dash='dash', line_color='gray', row=2, col=1)

# --- Bull Power ---
fig.add_trace(go.Scatter(
    x=df.index, y=df['Bull_Power'], mode='lines', name='Bull Power',
    line=dict(width=1.5, color='green')
), row=3, col=1)

fig.add_hline(y=BULL_LEVEL, line_dash='dash', line_color='gray', row=3, col=1)

# --- Stochastic ---
fig.add_trace(go.Scatter(
    x=df.index, y=df['Fast_%K'], mode='lines', name='Fast %K', line=dict(width=1.2, color='blue')
), row=4, col=1)

fig.add_trace(go.Scatter(
    x=df.index, y=df['Slow_%D'], mode='lines', name='Slow %D', line=dict(width=1.2, color='red')
), row=4, col=1)

fig.add_hline(y=STOCHASTIC_OVERBOUGHT, line_dash='dash', line_color='gray', row=4, col=1)
fig.add_hline(y=STOCHASTIC_OVERSOLD, line_dash='dash', line_color='gray', row=4, col=1)

# -------------------------
# Layout
# -------------------------

fig.update_layout(
    title=f"{symbol} Indicators Overview",
    xaxis_rangeslider_visible=False,
    template="plotly_white",
    height=900,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()


/tmp/ipython-input-2536304100.py:16: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed
